In [4]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
import optuna
from optuna.samplers import TPESampler
import joblib
import gc
import psutil
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Load data ─────────────────────────────────────────────
print("Loading data...")
df = pd.read_parquet('../data/processed/df_train.parquet')

# use last 2 years only
df = df[df['date'] >= '2014-01-01'].copy()
gc.collect()
print(f"Shape: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

# ── Split ─────────────────────────────────────────────────
TARGET   = 'sales'
EXCLUDE  = ['id', 'sales', 'date', 'wm_yr_wk']
FEATURES = [c for c in df.columns if c not in EXCLUDE]

cutoff_date = df['date'].max() - pd.Timedelta(days=28)
train = df[df['date'] <= cutoff_date]
val   = df[df['date'] >  cutoff_date]

X_train = train[FEATURES]
y_train = train[TARGET]
X_val   = val[FEATURES]
y_val   = val[TARGET]

del df, train, val
gc.collect()

print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"RAM available: {psutil.virtual_memory().available / 1e9:.1f} GB")

# ── LightGBM datasets ─────────────────────────────────────
dtrain = lgb.Dataset(
    X_train, label=y_train,
    categorical_feature=['item_id', 'dept_id', 'cat_id',
                         'store_id', 'state_id'],
    free_raw_data=True
)
dval = lgb.Dataset(
    X_val, label=y_val,
    reference=dtrain,
    free_raw_data=True
)

del X_train, y_train, X_val, y_val
gc.collect()
print(f"Datasets created!")
print(f"RAM available: {psutil.virtual_memory().available / 1e9:.1f} GB")

# ── Optuna objective ──────────────────────────────────────
def objective(trial):
    params = {
        'objective':              'tweedie',
        'tweedie_variance_power': 1.1,
        'metric':                 'rmse',
        'verbosity':              -1,
        'boosting_type':          'gbdt',
        'n_jobs':                 -1,
        'seed':                   42,
        'feature_pre_filter':     False, 
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 64, 512),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'subsample_freq':    1,
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.0, 1.0),
    }
    callbacks = [
        lgb.early_stopping(stopping_rounds=30, verbose=False),
        lgb.log_evaluation(period=-1)
    ]
    model = lgb.train(
        params, dtrain,
        num_boost_round=500,
        valid_sets=[dval],
        valid_names=['val'],
        callbacks=callbacks
    )
    return model.best_score['val']['rmse']

# ── Run Optuna ────────────────────────────────────────────
def save_callback(study, trial):
    joblib.dump(study, '../models/optuna_study.pkl')
    print(f"Trial {trial.number+1}/20 — RMSE: {trial.value:.4f} — Best: {study.best_value:.4f}")

import os
os.makedirs('../models', exist_ok=True)

print("\nStarting Optuna — 20 trials...")
sampler = TPESampler(seed=42)
study   = optuna.create_study(direction='minimize', sampler=sampler)
study.optimize(objective, n_trials=20, callbacks=[save_callback])

print(f"\n{'='*50}")
print(f"Best RMSE:   {study.best_value:.4f}")
print(f"Best params: {study.best_params}")
print(f"{'='*50}")

# ── Train final model ─────────────────────────────────────
best_params = study.best_params.copy()
best_params.update({
    'objective':              'tweedie',
    'tweedie_variance_power': 1.1,
    'metric':                 'rmse',
    'verbosity':              -1,
    'boosting_type':          'gbdt',
    'n_jobs':                 -1,
    'seed':                   42,
    'subsample_freq':         1,
})

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=True),
    lgb.log_evaluation(period=50)
]

final_model = lgb.train(
    best_params, dtrain,
    num_boost_round=1000,
    valid_sets=[dval],
    valid_names=['val'],
    callbacks=callbacks
)

final_rmse  = final_model.best_score['val']['rmse']
naive_rmse  = 2.2186
improvement = (naive_rmse - final_rmse) / naive_rmse * 100

print(f"\n{'='*50}")
print(f"Naive baseline: {naive_rmse:.4f}")
print(f"LightGBM:       {final_rmse:.4f}")
print(f"Improvement:    {improvement:.1f}%")
print(f"{'='*50}")

# ── Feature importance ────────────────────────────────────
importance = pd.DataFrame({
    'feature':    FEATURES,
    'importance': final_model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

print("\nTop 20 features:")
print(importance.head(20).to_string(index=False))

# ── Save ──────────────────────────────────────────────────
final_model.save_model('../models/lgbm_final.txt')
importance.to_csv('../models/feature_importance.csv', index=False)
print("\nDone! Model saved to ../models/")

Loading data...
Shape: (25764050, 50)
Memory: 3.68 GB
X_train: (24910330, 48)
X_val:   (853720, 48)
RAM available: 7.0 GB
Datasets created!
RAM available: 7.0 GB

Starting Optuna — 20 trials...
Trial 1/20 — RMSE: 1.8998 — Best: 1.8998
Trial 2/20 — RMSE: 1.9061 — Best: 1.8998
Trial 3/20 — RMSE: 1.9061 — Best: 1.8998
Trial 4/20 — RMSE: 1.9054 — Best: 1.8998
Trial 5/20 — RMSE: 1.9087 — Best: 1.8998
Trial 6/20 — RMSE: 1.9061 — Best: 1.8998
Trial 7/20 — RMSE: 1.9012 — Best: 1.8998
Trial 8/20 — RMSE: 1.9012 — Best: 1.8998
Trial 9/20 — RMSE: 1.9077 — Best: 1.8998
Trial 10/20 — RMSE: 1.9044 — Best: 1.8998
Trial 11/20 — RMSE: 1.9008 — Best: 1.8998
Trial 12/20 — RMSE: 1.9016 — Best: 1.8998
Trial 13/20 — RMSE: 1.9030 — Best: 1.8998
Trial 14/20 — RMSE: 1.9022 — Best: 1.8998
Trial 15/20 — RMSE: 1.8980 — Best: 1.8980
Trial 16/20 — RMSE: 1.8995 — Best: 1.8980
Trial 17/20 — RMSE: 1.9037 — Best: 1.8980
Trial 18/20 — RMSE: 1.9017 — Best: 1.8980
Trial 19/20 — RMSE: 1.9017 — Best: 1.8980
Trial 20/20 — RMS